# Steel Tank Thermal Enclosure Model

Kirchhoff-transform insulation sizing, API 650 shell, graphite piling thermal bridging, and foundation duct cooling.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# ── Physical constants ────────────────────────────────────────────────────────
sigma_sb = 5.6704e-8   # Stefan-Boltzmann [W/m2/K4]
R        = 8.314       # universal gas constant [J/mol/K]
g        = 9.81        # gravity [m/s2]

# ── Unit helpers ─────────────────────────────────────────────────────────────
ft_m  = 0.3048
in_m  = 0.0254
psi_Pa = 6894.76
MPa_Pa = 1e6


In [7]:
def k_thermax(T_C):
    """Thermax (calcium silicate) thermal conductivity [W/m/K] vs T in Celsius.
    Fit: linear interpolation through vendor data points.
    """
    pts = np.array([(20, 0.058), (200, 0.071), (400, 0.090),
                    (600, 0.116), (800, 0.150), (1000, 0.192)])
    return float(np.interp(T_C, pts[:,0], pts[:,1]))

def k_graphite_fn(T_C):
    """Graphite thermal conductivity [W/m/K]. Source: Mersen 2020 datasheet.
    Valid 20-2400 C.
    """
    _T = np.array([ 20, 200, 400, 600, 800,1000,1200,1400,1600,1800,2000,2200,2400])
    _k = np.array([115, 100,  85,  73,  62,  54,  49,  45,  42,  40,  39,  37,  34])
    return float(np.interp(np.clip(T_C, _T[0], _T[-1]), _T, _k))

def cp_graphite(T_C):
    """Graphite specific heat [J/kg/K]. Source: Entegris polynomial fit.
    T_C in degrees C; converts to K internally. Valid ~100-2500 C.
    """
    T_K = T_C + 273.15
    return (0.44391 + 0.30795e-4*T_K - 0.61257e5*T_K**(-2) + 0.10795e8*T_K**(-3)) * 4184

def air_props(T_K):
    """Air properties at temperature T_K (K).
    Returns dict: rho [kg/m3], mu [Pa s], k [W/m/K], Pr [-], cp [J/kg/K].
    """
    T_C = T_K - 273.15
    rho = 1.293 * 273.15 / T_K          # ideal gas, 1 atm
    mu  = 1.716e-5 * (T_K / 273.15)**0.686
    k   = 0.0241 * (T_K / 273.15)**0.82
    cp  = 1006 + 0.05 * T_C             # approx; [J/kg/K]
    Pr  = mu * cp / k
    return dict(rho=rho, mu=mu, k=k, Pr=Pr, cp=cp)

def kirchhoff_I(T_hot_K, T_cold_K, k_ins=None, eps1=0.2, eps2=0.2):
    """Kirchhoff transform heat flux intensity [W/m2] for combined radiation + conduction.

    Returns the integral I = integral_{T_cold}^{T_hot} [k(T) + 4*sigma*eps_eff*T^3] dT
    evaluated numerically.  eps_eff = 1/(1/eps1 + 1/eps2 - 1).

    If k_ins is None, uses k_thermax(T). Otherwise k_ins is a scalar [W/m/K].
    """
    eps_eff = 1.0 / (1.0/eps1 + 1.0/eps2 - 1.0)
    T_lo, T_hi = min(T_hot_K, T_cold_K), max(T_hot_K, T_cold_K)
    T_arr = np.linspace(T_lo, T_hi, 400)
    if k_ins is None:
        k_arr = np.array([k_thermax(T - 273.15) for T in T_arr])
    else:
        k_arr = np.full_like(T_arr, k_ins)
    integrand = k_arr + 4.0 * sigma_sb * eps_eff * T_arr**3
    return float(np.trapz(integrand, T_arr))

def h_nc_vertical(dT, H, T_film_K):
    """Natural convection h [W/m2/K] on a vertical surface, Churchill-Chu."""
    ap = air_props(T_film_K)
    if dT <= 0 or H <= 0:
        return 0.0
    Ra = g * (1.0/T_film_K) * abs(dT) * H**3 / (ap['mu']/ap['rho'])**2 * ap['Pr']
    Nu = (0.825 + 0.387*Ra**(1/6) / (1 + (0.492/ap['Pr'])**(9/16))**(8/27))**2
    return Nu * ap['k'] / H

def h_nc_horizontal_up(dT, L_char, T_film_K):
    """Natural convection h [W/m2/K], hot surface facing up (or cold facing down)."""
    ap = air_props(T_film_K)
    if dT <= 0 or L_char <= 0:
        return 0.0
    Ra = g * (1.0/T_film_K) * abs(dT) * L_char**3 / (ap['mu']/ap['rho'])**2 * ap['Pr']
    if Ra < 1e7:
        Nu = 0.54 * Ra**0.25
    else:
        Nu = 0.15 * Ra**(1/3)
    return Nu * ap['k'] / L_char


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# INPUTS
# ══════════════════════════════════════════════════════════════════════════════

# Storage block layout
annular       = False        # True = annular ring; False = solid cylinder
fill_ratio    = 0.70         # fraction of cylinder/annulus volume occupied by blocks

# Energy and temperature
E_MWh         = 3000.0        # stored energy [MWh]
T_hot_C       = 2400.0        # mean hot storage temperature [°C]
dT_swing_K    = 500.0        # charge/discharge swing [K]  (not used in steady-state sizing)

# Geometry
H_hot         = 10.0          # hot zone height [m]
r_inner_ann   = 0.0          # annular inner radius [m] (used only if annular=True)

# Ambient
T_amb_C       = 40.0         # ambient air temperature [°C]
I_solar_W     = 1400.0          # solar irradiance on tank wall [W/m2] (use 0 for indoor)

# Emissivities
eps_shell     = 0.3         # painted steel tank wall (exterior)
eps_concrete  = 0.9         # concrete pad (pad side)

# Insulation
loss_pct_day    = 1.5          # standby heat loss [%/day]  <- primary input
standby_frac    = loss_pct_day / 100.0   # fraction/day (derived)
thermax_bracket = 'low'      # Thermax k: 'low', 'high', or 'data' (measured, 640 kg/m3)
k_ins_wall    = None         # None → use k_thermax(T); or scalar [W/m/K]

# Air duct (forced convection under foundation)
D_duct        = 0.20         # duct hydraulic diameter [m]
n_duct        = 8            # number of parallel ducts
L_duct        = 20.0         # duct length [m]
eta_fan       = 0.85         # fan total efficiency [-]
T_duct_in_C   = 40.0         # inlet air temperature [°C]
q_floor_max   = 150.0        # max heat flux into ground [W/m2] — sets bottom insulation thickness

# API 650 shell sizing
S_d_psi       = 21000.0      # allowable shell stress [psi]  (A36/A285-C: 21000 psi typical)
G_fluid       = 1.0          # specific gravity of stored fluid (graphite pile → conservative 1.0)
CA_mm         = 1.6          # corrosion allowance [mm]
E_weld        = 1.0          # weld joint efficiency

# Graphite piling (bottom insulation structural support)
sigma_c_MPa   = 35.0         # graphite compressive strength [MPa]
SF_piling     = 3.0          # safety factor on compression
p_piling      = 0.5          # piling grid pitch [m]  (centre-to-centre)
d_piling      = 0.05         # piling diameter [m]

# Graphite block density
rho_graphite  = 1750.0       # [kg/m3]

# Thermax insulation cost — largest cost driver; replace with contract price
thermax_price_per_kg  = 2.50  # [$/kg]

# Column stack optimizer
graphite_price_per_kg = 8.0   # [$/kg] isostatic graphite
graphite_machining    = 1.3   # machining / handling multiplier
# Creep design life
design_life_yr        = 30    # [years] total design life at operating temperature
# Arrhenius creep law constants (engineer-supplied; eps_dot=A*(s/S)^n*exp(-Q/RT))
CREEP_A_arr    = 1.75         # pre-exponential [1/s]
CREEP_n_arr    = 3.8          # stress exponent [-]
CREEP_Q_cal    = 50000.0      # activation energy [cal/mol]
CREEP_Rcal     = 1.987204     # gas constant [cal/mol/K]
CREEP_S_Pa     = 98e6         # reference strength [Pa] (implied from data)
CREEP_eps_lim  = 0.01         # creep strain limit [-] (1%)
CREEP_yr_s     = 3.1536e7     # seconds per year [s/yr]
USE_ARRHENIUS  = True         # True=Arrhenius+cycle-weighting; False=tabulated KD
tank_cost_per_m       = 80e3  # [$/m] tank height penalty (shell steel estimate)

# ══════════════════════════════════════════════════════════════════════════════
# Derived temperatures
T_hot_K   = T_hot_C + 273.15
T_amb_K   = T_amb_C + 273.15
T_floor_K = T_amb_K          # ground temperature (conservative: same as ambient)
T_duct_in = T_duct_in_C + 273.15


In [12]:
# ── Storage block geometry ────────────────────────────────────────────────────
# E = rho * cp_avg * fill * V_block * dT_swing / 3.6e9  (MWh)
# Solve for V_block = V_hot * fill_ratio

T_avg_C     = T_hot_C - dT_swing_K / 2.0
cp_avg      = cp_graphite(T_avg_C)              # [J/kg/K]
rho_block   = rho_graphite * fill_ratio         # effective density in hot zone [kg/m3]

E_J         = E_MWh * 3.6e9                     # [J]
m_graphite  = E_J / (cp_avg * dT_swing_K)       # [kg]
V_graphite  = m_graphite / rho_graphite         # solid graphite volume [m3]
V_hot       = V_graphite / fill_ratio           # total hot zone volume [m3]
A_foot      = V_hot / H_hot                     # hot zone floor area [m2]

if annular:
    # A_foot = pi*(r_outer^2 - r_inner^2)
    r_inner = r_inner_ann
    r_outer = np.sqrt(r_inner**2 + A_foot / np.pi)
    D_tank  = 2 * r_outer
    C_shape = f"annular  r_in={r_inner:.2f} m  r_out={r_outer:.2f} m"
    n_edges = 4  # inner wall + outer wall + top + bottom (but lateral edges: just 2 circles)
else:
    r_inner = 0.0
    r_outer = np.sqrt(A_foot / np.pi)
    D_tank  = 2 * r_outer
    C_shape = f"cylinder  r={r_outer:.2f} m  D={D_tank:.2f} m"
    n_edges = 1  # single circular perimeter

C_tank = 2 * np.pi * r_outer   # outer circumference [m]
H_tank = H_hot                  # tank shell height ≈ hot zone height

print(f"Storage: {E_MWh:.0f} MWh   T_hot={T_hot_C:.0f} C   dT={dT_swing_K:.0f} K")
print(f"  m_graphite = {m_graphite/1000:.1f} t   V_hot = {V_hot:.1f} m3")
print(f"  Layout: {C_shape}")
print(f"  A_foot = {A_foot:.1f} m2   H_hot = {H_hot:.1f} m")


Storage: 3000 MWh   T_hot=2400 C   dT=500 K
  m_graphite = 10145.3 t   V_hot = 8281.9 m3
  Layout: cylinder  r=16.24 m  D=32.47 m
  A_foot = 828.2 m2   H_hot = 10.0 m


In [13]:
# ── Insulation thickness: wall and roof ───────────────────────────────────────
# Allowable standby loss
Q_standby_max = standby_frac * E_MWh * 1e6 / 24.0   # [W]  (over 24 h × fill fraction)

# Initial guesses
T_shell_guess = T_amb_C + 15.0   # [°C]

def _wall_losses(L_wall, L_roof, T_sh_C):
    """Total heat loss [W] through cylindrical wall + flat roof given shell temp."""
    T_sh_K = T_sh_C + 273.15
    I_wall = kirchhoff_I(T_hot_K, T_sh_K, k_ins=k_ins_wall)
    I_roof = kirchhoff_I(T_hot_K, T_sh_K, k_ins=k_ins_wall)

    # Cylindrical wall
    if annular:
        A_wall_in  = 2 * np.pi * r_inner * H_hot
        A_wall_out = 2 * np.pi * r_outer * H_hot
        # Use log-mean area for cylinder
        A_lm_wall  = (A_wall_out - A_wall_in) / np.log(A_wall_out / A_wall_in)
        Q_wall     = A_lm_wall * I_wall / L_wall
    else:
        A_wall_in  = 2 * np.pi * r_outer * H_hot   # approximate: treat as planar (thin ins)
        Q_wall     = A_wall_in * I_wall / L_wall

    # Roof (flat)
    Q_roof = A_foot * I_roof / L_roof

    # Edge correction (McQuiston 3D correction: 0.54 * L_edge * I)
    L_edge = C_tank
    Q_edge = 0.54 * L_edge * (I_wall + I_roof) / 2.0

    return Q_wall + Q_roof + Q_edge

def _shell_balance(T_sh_C, L_wall, L_roof):
    """Residual: heat out through insulation = heat dissipated from shell surface."""
    T_sh_K = T_sh_C + 273.15
    dT_sh  = T_sh_C - T_amb_C
    if dT_sh < 0:
        dT_sh = 0.0

    # Heat from insulation into shell interior surface  (per unit area → total)
    Q_in = _wall_losses(L_wall, L_roof, T_sh_C)

    # Shell exterior area (outer wall only for annular; includes roof)
    A_ext_wall = 2 * np.pi * r_outer * H_hot
    A_ext_roof = A_foot

    # Natural convection: vertical wall
    T_film = T_sh_K + 0.5 * (T_amb_K - T_sh_K)   # film temperature
    h_wall = h_nc_vertical(dT_sh, H_hot, T_film)
    h_roof = h_nc_horizontal_up(dT_sh, 2*r_outer, T_film)

    # Radiation to sky/ambient
    Q_rad_wall = eps_shell * sigma_sb * (T_sh_K**4 - T_amb_K**4) * A_ext_wall
    Q_rad_roof = eps_shell * sigma_sb * (T_sh_K**4 - T_amb_K**4) * A_ext_roof

    Q_conv_wall = h_wall * dT_sh * A_ext_wall
    Q_conv_roof = h_roof * dT_sh * A_ext_roof

    Q_solar     = I_solar_W * (A_ext_wall + A_ext_roof)   # absorbed solar (use abs=eps)

    Q_out = Q_rad_wall + Q_rad_roof + Q_conv_wall + Q_conv_roof + Q_solar

    return Q_in - Q_out   # should be zero at equilibrium

# ── Iterative solve for L_wall, L_roof, T_shell ───────────────────────────────
# Strategy: assume L_wall = L_roof (same insulation both sides), iterate T_shell.
# Then adjust L to hit Q_standby_max.

# Step 1: find T_shell for a given total loss budget.
# We need I(T_hot, T_shell) / L = q_wall_target
# Use equal thickness for wall and roof as first cut.

# Iterate: guess L, solve for T_shell, converge on wall loss = full standby budget.
# This is the minimum possible wall thickness (assumes no bottom losses).
# The column optimizer below accounts for bottom losses and finds the true split.
L_wall = 0.5   # initial guess [m]
L_roof = 0.5

for _iter in range(30):
    # Solve T_shell via brentq
    try:
        T_sh_sol = brentq(_shell_balance, T_amb_C + 0.1, T_hot_C - 1.0,
                          args=(L_wall, L_roof), xtol=0.01)
    except ValueError:
        T_sh_sol = T_amb_C + 20.0

    T_sh_K   = T_sh_sol + 273.15
    I_w      = kirchhoff_I(T_hot_K, T_sh_K, k_ins=k_ins_wall)

    Q_actual = _wall_losses(L_wall, L_roof, T_sh_sol)
    # Adjust L proportionally
    L_new    = L_wall * Q_actual / max(Q_standby_max, 1.0)
    if abs(L_new - L_wall) < 1e-4:
        L_wall = L_new; L_roof = L_new
        break
    L_wall = L_new; L_roof = L_new

T_shell_C  = T_sh_sol
Q_wall_act = _wall_losses(L_wall, L_roof, T_shell_C)

print(f"Wall / roof insulation thickness: {L_wall*100:.1f} cm  ({L_wall/in_m:.2f} in)")
print(f"Shell exterior temperature:        {T_shell_C:.1f} C")
print(f"Heat loss through wall + roof:     {Q_wall_act/1000:.1f} kW  ({Q_wall_act/Q_standby_max*100:.0f}% of budget)")
print(f"Total standby budget:              {Q_standby_max/1000:.1f} kW")
print(f"  Wall thickness is a lower bound — optimizer accounts for bottom losses.")


Wall / roof insulation thickness: 381385630884750052192040909799424.0 cm  (150151823182972456244640162512896.00 in)
Shell exterior temperature:        463.4 C
Heat loss through wall + roof:     17639.5 kW  (941% of budget)
Total standby budget:              1875.0 kW
  Wall thickness is a lower bound — optimizer accounts for bottom losses.


In [ ]:
# ── API 650 one-foot rule shell thickness ─────────────────────────────────────
# t [mm] = 4.9 * D [m] * (H - 0.3) [m] / (S_d [psi] * G * E) + CA [mm]
# D = tank diameter, H = liquid height (use H_hot conservatively)

D_m  = D_tank          # [m]
H_m  = H_hot           # [m]

t_shell_mm = 4.9 * D_m * max(H_m - 0.3, 0.0) / (S_d_psi * G_fluid * E_weld) + CA_mm
t_shell_mm = max(t_shell_mm, 6.0)   # API 650 minimum 6 mm for tanks > 15 m diameter; 5 mm otherwise

print(f"API 650 shell sizing:")
print(f"  D = {D_m:.2f} m   H = {H_m:.2f} m")
print(f"  t_shell = {t_shell_mm:.1f} mm  ({t_shell_mm/25.4:.3f} in)")
print(f"  S_d = {S_d_psi:.0f} psi   CA = {CA_mm:.1f} mm   E = {E_weld:.2f}")


In [ ]:
# ── Graphite piling structural sizing ────────────────────────────────────────
# Pilings carry weight of storage blocks through bottom insulation.
# Allowable compressive stress = sigma_c / SF

W_total_N      = m_graphite * g                    # total block weight [N]
sigma_allow_Pa = sigma_c_MPa * MPa_Pa / SF_piling  # [Pa]

A_pil_one      = np.pi * (d_piling/2)**2           # cross-section of one piling [m2]
A_pil_total_struct = W_total_N / sigma_allow_Pa    # total piling area needed [m2]

n_pilings_struct = int(np.ceil(A_pil_total_struct / A_pil_one))

# Grid layout within A_foot
n_pilings_grid = int(A_foot / p_piling**2)         # pilings from grid pitch
n_pilings      = max(n_pilings_struct, n_pilings_grid)

A_pil_total    = n_pilings * A_pil_one             # actual total piling area [m2]

print(f"Graphite pilings (bottom insulation support):")
print(f"  Weight = {W_total_N/1000:.0f} kN   sigma_allow = {sigma_allow_Pa/MPa_Pa:.1f} MPa")
print(f"  Required total A_pil = {A_pil_total_struct*1e4:.1f} cm2")
print(f"  d_piling = {d_piling*100:.0f} cm   A_one = {A_pil_one*1e4:.2f} cm2")
print(f"  n from strength:  {n_pilings_struct}")
print(f"  n from pitch {p_piling:.2f}m grid:  {n_pilings_grid}")
print(f"  n_pilings used = {n_pilings}")
print(f"  A_pil_total = {A_pil_total:.4f} m2  ({A_pil_total/A_foot*100:.1f} % of A_foot)")


In [ ]:
# ── Bottom insulation thickness (with piling conductive short-circuit) ────────
#
# Heat path through bottom:
#   - Through insulation around pilings:  Q_ins = (A_foot - A_pil) * I_bot / L_bot
#   - Through graphite pilings:           Q_pil = A_pil * k_avg * (T_hot - T_floor) / L_bot
#
# Constraint:  (Q_ins + Q_pil) / A_foot <= q_floor_max
#
# Solve for L_bot:
#   L_bot = [(A_foot - A_pil)*I_bot + A_pil*k_avg*(T_hot-T_floor)] / (q_floor_max * A_foot)

T_floor = T_floor_K   # ground temperature [K]

# Effective k of graphite over temperature range (hot face to floor)
k_graphite_avg = (k_graphite_fn(T_hot_C) + k_graphite_fn(T_amb_C)) / 2.0

# Kirchhoff integral for bottom insulation (hot face to floor)
I_bot = kirchhoff_I(T_hot_K, T_floor, k_ins=k_ins_wall)

numerator = ((A_foot - A_pil_total) * I_bot
           + A_pil_total * k_graphite_avg * (T_hot_K - T_floor))
L_bottom  = numerator / (q_floor_max * A_foot)

# Component heat fluxes at this thickness
Q_bot_ins  = (A_foot - A_pil_total) * I_bot / L_bottom
Q_bot_pil  = A_pil_total * k_graphite_avg * (T_hot_K - T_floor) / L_bottom
Q_bot_tot  = Q_bot_ins + Q_bot_pil

print(f"Bottom insulation:")
print(f"  I_bot = {I_bot:.1f} W/m   k_graphite_avg = {k_graphite_avg:.1f} W/m/K")
print(f"  L_bottom = {L_bottom*100:.1f} cm  ({L_bottom/in_m:.2f} in)")
print(f"  Q_bot_ins  = {Q_bot_ins/1000:.2f} kW  (through insulation)")
print(f"  Q_bot_pil  = {Q_bot_pil/1000:.2f} kW  (through graphite pilings)")
print(f"  Q_bot_tot  = {Q_bot_tot/1000:.2f} kW  (total)")
print(f"  q_floor    = {Q_bot_tot/A_foot:.1f} W/m2  (limit = {q_floor_max:.0f} W/m2)")


In [ ]:
# ── Foundation forced-air duct cooling ────────────────────────────────────────
# Size fan to remove Q_bot_tot from foundation via forced convection in ducts.

T_duct_out_guess = T_duct_in + Q_bot_tot / (n_duct * 1.2 * 1006 * 0.5)  # first guess

def _duct_solve(mdot_per_duct):
    """Given mass flow per duct, compute T_out, h, dP, fan power."""
    ap_in  = air_props(T_duct_in)
    T_film = T_duct_in + Q_bot_tot / (2 * n_duct * mdot_per_duct * ap_in['cp'])
    ap     = air_props(T_film)

    Re  = 4 * mdot_per_duct / (np.pi * D_duct * ap['mu'])
    Pr  = ap['Pr']

    if Re < 2300:
        Nu = 3.66   # laminar, fully developed
    else:
        f   = (0.790 * np.log(Re) - 1.64)**(-2)   # Petukhov friction factor
        Nu  = (f/8) * Re * Pr / (1.07 + 12.7*(f/8)**0.5 * (Pr**(2/3) - 1))

    h_duct = Nu * ap['k'] / D_duct   # [W/m2/K]
    A_duct = np.pi * D_duct * L_duct  # [m2]

    NTU   = h_duct * A_duct / (mdot_per_duct * ap['cp'])
    eff   = 1 - np.exp(-NTU)
    Q_per_duct = eff * mdot_per_duct * ap['cp'] * (T_floor - T_duct_in)
    # Note: T_floor is "hot" side here; if Q_bot_pil heats the slab above ambient use T_bot_slab
    T_out  = T_duct_in + Q_per_duct / (mdot_per_duct * ap['cp'])

    # Duct pressure drop (Petukhov)
    dP_fric = f * (L_duct / D_duct) * 0.5 * ap['rho'] * (4*mdot_per_duct/(ap['rho']*np.pi*(D_duct/2)**2))**2
    dP_minor = 1.5 * 0.5 * ap['rho'] * (4*mdot_per_duct/(ap['rho']*np.pi*(D_duct/2)**2))**2
    dP_tot = dP_fric + dP_minor
    W_fan  = mdot_per_duct / ap['rho'] * dP_tot / eta_fan

    return dict(T_out=T_out, h=h_duct, Re=Re, Nu=Nu, dP=dP_tot,
                W_fan=W_fan, Q_per_duct=Q_per_duct)

# Find mdot that delivers Q_bot_tot total across all ducts
from scipy.optimize import brentq as _brentq

def _residual(mdot):
    d = _duct_solve(mdot)
    return n_duct * d['Q_per_duct'] - Q_bot_tot

try:
    mdot_sol = _brentq(_residual, 1e-4, 10.0, xtol=1e-5)
except ValueError:
    mdot_sol = 0.5   # fallback

d = _duct_solve(mdot_sol)
W_fan_total = n_duct * d['W_fan']
mdot_total  = n_duct * mdot_sol

print(f"Foundation duct cooling:")
print(f"  Q to remove = {Q_bot_tot/1000:.2f} kW  across {n_duct} ducts  (D={D_duct*100:.0f} cm, L={L_duct:.0f} m)")
print(f"  mdot total  = {mdot_total:.2f} kg/s  ({mdot_sol:.3f} kg/s per duct)")
print(f"  Re = {d['Re']:.0f}   Nu = {d['Nu']:.1f}   h = {d['h']:.1f} W/m2/K")
print(f"  T_out = {d['T_out']-273.15:.1f} C   dP/duct = {d['dP']:.1f} Pa")
print(f"  Fan power total = {W_fan_total:.1f} W  ({W_fan_total/1000:.3f} kW)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY REPORT
# ══════════════════════════════════════════════════════════════════════════════

Q_total_kW = (Q_wall_act + Q_bot_tot) / 1000.0

print("=" * 65)
print(f"  STEEL TANK THERMAL ENCLOSURE — SIZING SUMMARY")
print("=" * 65)
print(f"  Layout:          {C_shape}")
print(f"  Energy stored:   {E_MWh:.0f} MWh   ({m_graphite/1000:.1f} t graphite)")
print(f"  T_hot:           {T_hot_C:.0f} C     dT_swing: {dT_swing_K:.0f} K")
print()
print(f"  INSULATION")
print(f"    Wall / roof:   {L_wall*100:.1f} cm  ({L_wall/in_m:.2f} in)")
print(f"    Bottom:        {L_bottom*100:.1f} cm  ({L_bottom/in_m:.2f} in)")
print()
print(f"  SHELL")
print(f"    D_tank:        {D_tank:.2f} m   H_tank: {H_tank:.2f} m")
print(f"    t_shell:       {t_shell_mm:.1f} mm   (API 650)")
print(f"    T_shell:       {T_shell_C:.1f} C  (exterior surface)")
print()
print(f"  HEAT LOSSES")
print(f"    Wall + roof:   {Q_wall_act/1000:.1f} kW")
print(f"    Bottom total:  {Q_bot_tot/1000:.1f} kW  "
      f"(ins {Q_bot_ins/1000:.1f} kW + pilings {Q_bot_pil/1000:.1f} kW)")
print(f"    TOTAL:         {Q_total_kW:.1f} kW")
print(f"    q_floor:       {Q_bot_tot/A_foot:.1f} W/m2  (limit {q_floor_max:.0f} W/m2)")
standby_pct = Q_total_kW * 86.4 / (E_MWh * 1000) * 100
print(f"    Standby loss:  {standby_pct:.2f} %/day   (target ≤ {standby_frac*100:.1f} %/day)")
print()
print(f"  GRAPHITE PILINGS")
print(f"    n_pilings:     {n_pilings}   d={d_piling*100:.0f} cm   pitch={p_piling:.2f} m")
print(f"    A_pil/A_foot:  {A_pil_total/A_foot*100:.1f} %")
print()
print(f"  FOUNDATION COOLING")
print(f"    Fan power:     {W_fan_total:.1f} W   mdot: {mdot_total:.2f} kg/s")
print("=" * 65)


In [ ]:
# ── Material database (bottom stack optimizer) ────────────────────────────────
from scipy.integrate import quad as _quad
import itertools

# Thermax k polynomial bracket: k(T_K) = a + b*T^3  [W/m/K]
# Analytic Kirchhoff integral: I(T1,T2) = a*(T1-T2) + b/4*(T1^4-T2^4)
TH_RHO      = 550.0   # bracket case packed density [kg/m3]
TH_RHO_DATA = 640.0   # measured density for 'data' case [kg/m3]
_TH_BRACKET = {'low': (0.030, 1.0e-11), 'high': (0.050, 2.0e-11)}
# Measured Thermax fit (thermax_data.py): k = k0*exp(bb*T_C) at 640 kg/m3
_K0_DATA = 0.04916; _BB_DATA = 0.0011

def kth(T_K, bracket='low'):
    if bracket == 'data':
        return _K0_DATA * np.exp(_BB_DATA * (T_K - 273.15))
    a, b = _TH_BRACKET[bracket]
    return a + b * T_K**3

def I_thermax(T1_K, T2_K, bracket='low'):
    """Analytic Kirchhoff integral over a Thermax layer [W/m]."""
    if bracket == 'data':
        return (kth(T1_K, 'data') - kth(T2_K, 'data')) / _BB_DATA
    a, b = _TH_BRACKET[bracket]
    return a*(T1_K - T2_K) + b/4.0*(T1_K**4 - T2_K**4)

def th_rho(bracket):
    """Thermax packed density [kg/m3] for given bracket."""
    return TH_RHO_DATA if bracket == 'data' else TH_RHO

# name -> (T_max_K, k_fn(T_K), rho [kg/m3], $/kg, sigma_allow_Pa or None=full-area)
# T_max: safe limit in contact with carbon in argon (kinetic/chemical limit)
MATERIALS = {
    'tech_alumina':   (1573, lambda T: 30.0*(300/T)**1.1, 3900, 25.0, 100e6),
    'high_al_refr':   (1473, lambda T: 2.8 + 1000.0/T,   3000,  2.5,  10e6),
    'zirconia_psz':   (1373, lambda T: 2.2,               5900, 40.0, 100e6),
    'fused_silica':   (1273, lambda T: 1.2,               1950,  5.0,   6e6),
    'dense_castable': (1173, lambda T: 1.8,               2700,  1.5,   6e6),
    'ins_castable':   (1173, lambda T: 0.35,              1200,  1.2,   None),
}
_MAT_LABEL = {
    'tech_alumina':   'Alumina 99.5%      (<=1300C)',
    'high_al_refr':   'High-Al refractory (<=1200C)',
    'zirconia_psz':   'Zirconia PSZ/YSZ   (<=1100C)',
    'fused_silica':   'Fused silica       (<=1000C)',
    'dense_castable': 'Dense CAC castable  (<=900C)',
    'ins_castable':   'Ins. castable full  (<=900C)',
}

print(f"Thermax: ${thermax_price_per_kg:.2f}/kg")
for br in ['low', 'high', 'data']:
    rho_br = th_rho(br)
    print(f"  [{br:4s}] rho={rho_br:.0f} kg/m3  "
          f"k(300K)={kth(300,br)*1000:.1f} mW/m/K  "
          f"k(2673K)={kth(2673,br):.3f} W/m/K  "
          f"I(2673->330K)={I_thermax(2673,330,br):.0f} W/m  "
          f"cost={thermax_price_per_kg*rho_br:.0f} $/m3")
print()
print(f"{'Material':<36}  {'T_max':>7}  {'k@1000K':>8}  {'rho':>5}  {'$/kg':>5}  {'sigma':>10}")
for nm, (Tm, kf, rho, pr, sg) in MATERIALS.items():
    print(f"  {_MAT_LABEL[nm]:<34}  {Tm-273:>5.0f}C  {kf(1000):>8.2f}  "
          f"{rho:>5.0f}  {pr:>5.2f}  "
          f"{'full area' if sg is None else str(int(sg/1e6))+' MPa':>10}")


In [ ]:
# ── Graphite column creep model ───────────────────────────────────────────────
# Bearing pressure (graphite stack weight + 10 kPa misc)
p_bearing = fill_ratio * rho_graphite * g * H_hot + 10e3   # [Pa]

# Graphite material properties
G_SIG_RT  = 100e6   # RT compressive strength [Pa], isostatic grade
G_SF_RT   = 6       # safety factor for strength-governed (low-T) zone
F_MIN_COL = 0.004   # minimum column area fraction (buckling/seismic floor)

# ── Creep knockdown table ─────────────────────────────────────────────────────
# Fraction of RT strength (sigma_allow / sigma_RT) to keep creep < 1%
# at each temperature, for 1-year and 10-year exposure.
# T in degrees C, 1000-2500 C in 100 C steps.
_CREEP_T_C  = np.arange(1000, 2501, 100, dtype=float)
_CREEP_KD1  = np.array([0.495, 0.339, 0.245, 0.184, 0.143, 0.114, 0.094, 0.078,
                         0.067, 0.057, 0.050, 0.044, 0.040, 0.036, 0.033, 0.030])
_CREEP_KD10 = np.array([0.270, 0.185, 0.133, 0.100, 0.078, 0.062, 0.051, 0.043,
                         0.036, 0.031, 0.027, 0.024, 0.022, 0.020, 0.018, 0.016])

# Effective stress exponent n_eff(T) derived from 1-yr / 10-yr ratio:
#   KD(t) = KD_1yr * t^(-1/n_eff)  =>  n_eff = log(10) / log(KD_1yr / KD_10yr)
_CREEP_N = np.log(10) / np.log(_CREEP_KD1 / _CREEP_KD10)

def _kd_at_life(T_C, t_yr):
    """Knockdown factor KD = sigma_allow / sigma_RT at temperature T_C for t_yr years.
    Interpolates table data; extrapolates in time using local power-law exponent.
    Below 1000 C: returns 1.0 (strength governs, no creep knockdown).
    """
    T_C_clip = np.clip(T_C, _CREEP_T_C[0], _CREEP_T_C[-1])
    kd1  = float(np.interp(T_C_clip, _CREEP_T_C, _CREEP_KD1))
    kd10 = float(np.interp(T_C_clip, _CREEP_T_C, _CREEP_KD10))
    n    = float(np.interp(T_C_clip, _CREEP_T_C, _CREEP_N))
    if T_C < _CREEP_T_C[0]:
        return 1.0   # below table: strength governs
    t = max(t_yr, 0.01)
    # anchor on 10-yr point to avoid extrapolating from 1-yr base too far
    return kd10 * (t / 10.0)**(-1.0 / n)

def sigma_creep_allow(T_K):
    """Creep-limited compressive allowable [Pa] at T_K for design_life_yr."""
    kd = _kd_at_life(T_K - 273.15, design_life_yr)
    return min(kd * G_SIG_RT, G_SIG_RT / G_SF_RT)

def f_graphite_col(T_K):
    """Graded column area fraction at temperature T_K."""
    return max(F_MIN_COL, p_bearing / sigma_creep_allow(T_K))

def kg_col(T_K):
    """Graphite column thermal conductivity [W/m/K]. Mersen 2020."""
    return k_graphite_fn(T_K - 273.15)

# ── Table and plots ───────────────────────────────────────────────────────────
print(f"Bearing pressure: {p_bearing/1e3:.1f} kPa")
print(f"Design life: {design_life_yr} years")
print()
print(f"{'T [C]':>7}  {'n_eff':>6}  {'KD_1yr':>7}  {'KD_10yr':>8}  "
      f"{'KD_{yr}yr'.replace('{yr}', str(design_life_yr)):>10}  "
      f"{'sigma_allow [MPa]':>18}  {'f_col [%]':>9}  {'k_g [W/m/K]':>11}")
for T_C in list(np.arange(1000, 2501, 200)) + [T_hot_K - 273.15]:
    T_K = T_C + 273.15
    kd  = _kd_at_life(T_C, design_life_yr)
    sig = sigma_creep_allow(T_K)
    kd1  = float(np.interp(np.clip(T_C, 1000, 2500), _CREEP_T_C, _CREEP_KD1))
    kd10 = float(np.interp(np.clip(T_C, 1000, 2500), _CREEP_T_C, _CREEP_KD10))
    n_v  = float(np.interp(np.clip(T_C, 1000, 2500), _CREEP_T_C, _CREEP_N))
    print(f"  {T_C:>5.0f}  {n_v:>6.2f}  {kd1:>7.3f}  {kd10:>8.3f}  "
          f"{kd:>10.4f}  {sig/1e6:>18.2f}  "
          f"{f_graphite_col(T_K)*100:>9.3f}  {kg_col(T_K):>11.1f}")

# Plots
_Tv = np.linspace(600, T_hot_K, 120)
_kd_life = np.array([_kd_at_life(T-273.15, design_life_yr) for T in _Tv])
_sig_allow = np.array([sigma_creep_allow(T)/1e6 for T in _Tv])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.plot(_Tv-273, _kd_life, 'b-', lw=2, label=f'{design_life_yr} yr')
ax.plot(_CREEP_T_C, _CREEP_KD1,  'g--', lw=1.5, label='1 yr (table)')
ax.plot(_CREEP_T_C, _CREEP_KD10, 'r--', lw=1.5, label='10 yr (table)')
ax.set(xlabel='Temperature [C]', ylabel='Knockdown factor [-]',
       title='Creep knockdown KD(T, t)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(_Tv-273, _sig_allow, 'r-', lw=2)
ax.axhline(G_SIG_RT/G_SF_RT/1e6, color='gray', ls='--', lw=1,
           label=f'Strength limit {G_SIG_RT/G_SF_RT/1e6:.0f} MPa (SF={G_SF_RT})')
ax.set(xlabel='Temperature [C]', ylabel='sigma_allow [MPa]',
       title=f'Creep allowable ({design_life_yr} yr)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[2]
_fcol = np.array([f_graphite_col(T)*100 for T in _Tv])
ax.plot(_Tv-273, _fcol, 'b-', lw=2)
ax.axhline(F_MIN_COL*100, color='gray', ls='--', lw=1, label=f'F_MIN={F_MIN_COL*100:.1f}%')
ax.set(xlabel='Temperature [C]', ylabel='Column area fraction [%]',
       title='Graded column f(T)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# ── Arrhenius creep model with cycle weighting ────────────────────────────
# eps_dot = A*(sigma/S)^n*exp(-Q/RT)
# Cycle-weighted t_eq: w = mean(exp(-Q/R*(1/T - 1/T_hot))) over T_hot-dT..T_hot
_T_cyc = np.linspace(T_hot_K - dT_swing_K, T_hot_K, 201)
_w_cyc = float(np.mean(
    np.exp(-CREEP_Q_cal / CREEP_Rcal * (1.0/_T_cyc - 1.0/T_hot_K))))
_t_eq  = design_life_yr * _w_cyc   # effective years at peak temperature

def sigma_creep_arr(T_K, t_yr):
    """Arrhenius allowable [Pa] at T_K for t_yr yr exposure."""
    exp_t = np.exp(-CREEP_Q_cal / (CREEP_Rcal * T_K))
    s_cr  = CREEP_S_Pa * (
        CREEP_eps_lim / (CREEP_A_arr * t_yr * CREEP_yr_s * exp_t)
    )**(1.0 / CREEP_n_arr)
    return min(float(s_cr), G_SIG_RT / G_SF_RT)

def sigma_creep_allow_arr(T_K):
    """Arrhenius allowable with cycle-weighted equivalent life [Pa]."""
    return sigma_creep_arr(T_K, _t_eq)

def f_graphite_col_arr(T_K):
    return max(F_MIN_COL, p_bearing / sigma_creep_allow_arr(T_K))

# Override active model if USE_ARRHENIUS
if USE_ARRHENIUS:
    sigma_creep_allow = sigma_creep_allow_arr
    f_graphite_col    = f_graphite_col_arr

# Derived peak allowable
sigma_cr_peak_Pa = sigma_creep_allow(T_hot_K)
print(f"Arrhenius creep: cycle-weight w={_w_cyc:.3f}  "
      f"t_eq={_t_eq:.1f} yr  (USE_ARRHENIUS={USE_ARRHENIUS})")
print(f"  sigma_allow at T_hot ({T_hot_K-273:.0f}C): {sigma_cr_peak_Pa/1e6:.3f} MPa  "
      f"f_col={f_graphite_col(T_hot_K)*100:.1f}%")
print(f"  k_graphite at T_hot ({T_hot_K-273:.0f}C): {kg_col(T_hot_K):.1f} W/m/K")


In [ ]:
# ── Composite layer Kirchhoff integrals ───────────────────────────────────────
# Parallel conduction: k_eff(T) = f(T)*k_col(T) + (1-f(T))*kth(T, bracket)
# Layer thickness = I_eff / q_b;  volumes per unit footprint = J / q_b

def _layer_integ(Ta_K, Tb_K, k_col_fn, f_col_fn, bracket, full_area=False):
    """Kirchhoff integrals for one composite layer (Ta > Tb) [W/m].
    Returns (I_eff, col_J, th_J):
      I_eff = integral of k_eff dT  (sets thickness)
      col_J = integral of f*k_eff dT  (column material volume driver)
      th_J  = I_eff - col_J          (Thermax volume driver)
    """
    if Ta_K <= Tb_K + 1e-6:
        return 0.0, 0.0, 0.0
    if full_area:
        I = _quad(k_col_fn, Tb_K, Ta_K, limit=200)[0]
        return I, I, 0.0
    def keff(T):
        fc = f_col_fn(T)
        return fc * k_col_fn(T) + (1.0 - fc) * kth(T, bracket)
    I_eff = _quad(keff, Tb_K, Ta_K, limit=200)[0]
    col_J = _quad(lambda T: f_col_fn(T) * keff(T), Tb_K, Ta_K, limit=200)[0]
    return I_eff, col_J, I_eff - col_J

def _wall_loss_opt(L, bracket, Thot_K=None):
    """Heat loss [W]: cylindrical wall (log-mean) + flat roof + 3D edge.
    Q_cyl  = 2*pi*H*I / ln(R/r)     (R = r_outer + L)
    Q_roof = A_foot * I / L
    Q_edge = 0.54 * 2*pi*r * I * 1.5   (L-independent, McQuiston)
    """
    if Thot_K is None: Thot_K = T_hot_K
    Iw = I_thermax(Thot_K, T_amb_K, bracket)
    R  = r_outer + L
    Q  = 2*np.pi*H_hot * Iw / np.log(R / r_outer)   # cylindrical wall
    Q += A_foot * Iw / L                              # flat roof
    Q += 0.54 * 2*np.pi*r_outer * Iw * 1.5           # 3D top-edge correction
    if annular:
        R_in = max(r_inner - L, 1e-3)
        Q += 2*np.pi*H_hot * Iw / np.log(r_inner / R_in)  # inner wall
    return Q

def _wall_cycle_avg_factor(bracket):
    """Cycle-average I / peak I for Thermax walls (hot face swings dT_swing_K).
    Used to convert peak wall loss to cycle-average for budget comparison.
    """
    T_pts = np.linspace(T_hot_K - dT_swing_K, T_hot_K, 26)
    I_avg = float(np.mean([I_thermax(T, T_amb_K, bracket) for T in T_pts]))
    I_pk  = I_thermax(T_hot_K, T_amb_K, bracket)
    return I_avg / I_pk if I_pk > 0 else 1.0

def _wall_vol(L):
    """Thermax volume in walls + roof [m3]."""
    V = np.pi*((r_outer+L)**2 - r_outer**2)*H_hot + A_foot*L
    if annular:
        R_in = max(r_inner - L, 0.0)
        V += np.pi*(r_inner**2 - R_in**2)*H_hot
    return V

# ── Integral decomposition ────────────────────────────────────────────────────
T_gox = 1473.0   # approximate graphite-to-oxide transition [K]
print(f"Integral decomposition ({T_hot_K-273:.0f} -> {T_gox-273:.0f} C  |  {T_gox-273:.0f} -> {T_amb_C:.0f} C):")
for br in ['low', 'high']:
    Ig, _, _ = _layer_integ(T_hot_K, T_gox, kg_col, f_graphite_col, br)
    Ith_hot  = I_thermax(T_hot_K, T_gox, br)
    Ith_cold = I_thermax(T_gox, T_amb_K, br)
    print(f"  [{br}]  graphite cols: {Ig:.0f} W/m  |  Thermax hot zone: {Ith_hot:.0f} W/m  |  "
          f"Thermax cold zone: {Ith_cold:.0f} W/m")


In [ ]:
# ── Column stack optimizer ────────────────────────────────────────────────────
# Layers hot->cold: graphite | oxide-A | oxide-B  (A and B optional)
# Grids: T1 = graphite/A interface, T2 = A/B interface, q_b (bottom flux)
# Minimizes: column cost + Thermax cost (walls+bottom) + tank height surcharge

T_COLD    = T_amb_K    # cold face [K]
CYCLE_AVG = 0.87       # cycle-average loss factor (~0.87 of peak)
G_COST_M3 = graphite_price_per_kg * graphite_machining * 1800.0   # $/m3

def _th_cost_m3(bracket='low'):
    return thermax_price_per_kg * th_rho(bracket)   # $/m3

def _eval(T1_K, T2_K, mat_A, mat_B, q_b, bracket):
    TH_M3 = _th_cost_m3(bracket)
    layers = []   # (name, I_eff, col_J, th_J, col_cost_per_m3)

    Ig, cg, tg = _layer_integ(T_hot_K, T1_K, kg_col, f_graphite_col, bracket)
    layers.append(('graphite', Ig, cg, tg, G_COST_M3))

    if mat_A is not None:
        Tm, kf, rho, pr, sg = MATERIALS[mat_A]
        if T1_K > Tm + 1e-6: return None
        full = sg is None
        fa = (lambda T: 1.0) if full else (lambda T, s=sg: max(F_MIN_COL, p_bearing/s))
        IA, cA, tA = _layer_integ(T1_K, T2_K, kf, fa, bracket, full_area=full)
        layers.append((mat_A, IA, cA, tA, rho*pr))

    if mat_B is not None:
        Tm, kf, rho, pr, sg = MATERIALS[mat_B]
        if T2_K > Tm + 1e-6: return None
        full = sg is None
        fb = (lambda T: 1.0) if full else (lambda T, s=sg: max(F_MIN_COL, p_bearing/s))
        IB, cB, tB = _layer_integ(T2_K, T_COLD, kf, fb, bracket, full_area=full)
        layers.append((mat_B, IB, cB, tB, rho*pr))

    I_bot = sum(l[1] for l in layers)
    if I_bot <= 0: return None
    L_bot = I_bot / q_b

    loss_budget = E_MWh * 1e6 * standby_frac / 24.0   # W  (=Q_standby_max)
    Q_bot_avg   = CYCLE_AVG * q_b * A_foot
    Q_wall_bud  = loss_budget - Q_bot_avg
    if Q_wall_bud <= 1.0: return None
    fav_w = _wall_cycle_avg_factor(bracket)
    try:
        L_wall = brentq(
            lambda L: _wall_loss_opt(L, bracket) * fav_w - Q_wall_bud,
            0.01, 30.0, xtol=1e-3)
    except ValueError:
        return None

    cost_col  = sum(l[2]/q_b * A_foot * l[4] for l in layers)
    cost_th_b = sum(l[3]/q_b * A_foot for l in layers) * TH_M3
    cost_th_w = _wall_vol(L_wall) * TH_M3
    cost_tank = tank_cost_per_m * (L_bot + L_wall)
    total     = cost_col + cost_th_b + cost_th_w + cost_tank

    return dict(total=total, cost_col=cost_col, cost_th_b=cost_th_b,
                cost_th_w=cost_th_w, cost_tank=cost_tank,
                L_bot=L_bot, L_wall=L_wall, q_b=q_b, T1=T1_K, T2=T2_K,
                mat_A=mat_A, mat_B=mat_B, bracket=bracket,
                layers=[(l[0], l[1]/q_b) for l in layers])

def _optimize(bracket='low', q_b_vals=None):
    if q_b_vals is None:
        q_b_vals = [150, 200, 250, 300, 350, 400]
    mnames = list(MATERIALS.keys())
    combos = [(None, None)]
    for a in mnames:
        combos.append((a, None))
        for b in mnames:
            if MATERIALS[b][0] <= MATERIALS[a][0]:
                combos.append((a, b))

    results = []
    for mat_A, mat_B in combos:
        best = None
        T1s = np.arange(800, 1501, 25)+273.15 if mat_A else [T_COLD]
        for T1 in T1s:
            if mat_A and T1 > MATERIALS[mat_A][0]+1e-6: continue
            T2_hi = min(int(T1-273), 800)
            T2s = np.arange(400, T2_hi+1, 50)+273.15 if mat_B else [T_COLD]
            for T2 in T2s:
                if mat_B and T2 > MATERIALS[mat_B][0]+1e-6: continue
                if mat_B and T2 >= T1-1: continue
                for qb in q_b_vals:
                    r = _eval(T1, T2, mat_A, mat_B, qb, bracket)
                    if r and (best is None or r['total'] < best['total']):
                        best = r
        if best: results.append(best)
    results.sort(key=lambda r: r['total'])
    return results

_SHORT = {'graphite':'Gr', 'tech_alumina':'Al99', 'high_al_refr':'HA',
          'zirconia_psz':'ZrO2', 'fused_silica':'FS',
          'dense_castable':'DCast', 'ins_castable':'ICast'}

def _show(results, n=8):
    _bud = E_MWh * 1e6 * standby_frac / 24.0   # W
    W = 118
    print('─'*W)
    print(f"  {'Stack':<40}  {'Total':>8}  {'Col':>7}  {'ThBot':>7}  "
          f"{'ThWall':>7}  {'Tank':>7}  {'L_bot':>6}  {'L_wall':>6}  "
          f"{'q_b':>5}  {'bot%':>5}  {'wal%':>5}")
    print('─'*W)
    for r in results[:n]:
        segs = ' -> '.join(f"{_SHORT.get(nm,nm[:6])} {L:.2f}m" for nm, L in r['layers'])
        Q_bot = CYCLE_AVG * r['q_b'] * A_foot
        bot_pct = 100.0 * Q_bot / _bud
        wal_pct = 100.0 - bot_pct
        print(f"  {segs:<40}  ${r['total']/1e6:>6.2f}M  ${r['cost_col']/1e6:>5.2f}M  "
              f"${r['cost_th_b']/1e6:>5.2f}M  ${r['cost_th_w']/1e6:>5.2f}M  "
              f"${r['cost_tank']/1e6:>5.2f}M  {r['L_bot']:>5.2f}m  "
              f"{r['L_wall']:>5.2f}m  {r['q_b']:>5.0f}  "
              f"{bot_pct:>5.1f}%  {wal_pct:>5.1f}%")

# ── Run ───────────────────────────────────────────────────────────────────────
print(f"Optimizer: E={E_MWh:.0f} MWh  T_hot={T_hot_K-273:.0f}C  "
      f"loss<={standby_frac*100:.1f}%/day  Thermax ${thermax_price_per_kg:.2f}/kg  "
      f"graphite ${graphite_price_per_kg:.0f}/kg x{graphite_machining}  "
      f"sigma_cr={sigma_cr_peak_Pa/1e6:.1f} MPa")
print(f"  A_foot={A_foot:.1f} m2  r_outer={r_outer:.2f} m  H={H_hot:.1f} m  "
      f"p_bearing={p_bearing/1e3:.1f} kPa")

# ── Run ───────────────────────────────────────────────────────────────────
print(f"Optimizer: E={E_MWh:.0f} MWh  T_hot={T_hot_K-273:.0f}C  "
      f"loss<={loss_pct_day:.1f}%/day  Thermax ${thermax_price_per_kg:.2f}/kg  "
      f"graphite ${graphite_price_per_kg:.0f}/kg x{graphite_machining}  "
      f"sigma_cr={sigma_cr_peak_Pa/1e6:.2f} MPa  USE_ARRHENIUS={USE_ARRHENIUS}")
print(f"  A_foot={A_foot:.1f} m2  r_outer={r_outer:.2f} m  H={H_hot:.1f} m  "
      f"p_bearing={p_bearing/1e3:.1f} kPa")

_Q_B_GRID = [200, 300, 400, 500, 600, 800, 1000, 1200, 1600, 2000]

OPT_RESULTS = {}
for _br in ['low', 'high', 'data']:
    print(f"\n=== Thermax k: {_br} (rho={th_rho(_br):.0f} kg/m3, "
          f"fav_w={_wall_cycle_avg_factor(_br):.3f}) ===")
    OPT_RESULTS[_br] = _optimize(_br, q_b_vals=_Q_B_GRID)
    _show(OPT_RESULTS[_br], n=6)

_best_all = min((r for rs in OPT_RESULTS.values() for r in rs), key=lambda r: r["total"])
# Also expose best for the selected thermax_bracket
_best_bracket = OPT_RESULTS.get(thermax_bracket, [None])
_best_sel = _best_bracket[0] if _best_bracket else _best_all
_stack_str = " -> ".join(f"{nm} {L:.2f} m" for nm, L in _best_all["layers"])
print("\nBest overall: ${:.2f}M  [{}]".format(_best_all["total"]/1e6, _best_all["bracket"]))
print("  Stack:", _stack_str)
print("  L_bot={:.2f} m  L_wall={:.2f} m  q_b={:.0f} W/m2".format(
      _best_all["L_bot"], _best_all["L_wall"], _best_all["q_b"]))


In [ ]:
# ── Finned steel floor foundation cooling (floor.py — handoff update #3) ──────
# Forced air in channels under a finned steel floor plate.
# Geometry: 0.75 m rail pitch, 0.15 m rail width → 0.60 m channel width, 0.20 m deep.
# Fins: 10 mm thick × 170 mm tall @ 100 mm pitch, 6 fins/channel, steel k=45 W/m/K.
# Flow: perimeter inlets → L=19 m to central exhaust trench → fans → stack.
# Design day: T_amb_C inlet, plate limit 93°C (API 650 base plate max).

_rho_a = 1.12; _cp_a = 1007; _nu_a = 1.8e-5; _ka = 0.028; _Pr_a = 0.71; _kst = 45.0
_rail_p = 0.75; _rail_w = 0.15; _ch_w = _rail_p - _rail_w   # channel width 0.60 m
_H_ch   = 0.20; _Hf = 0.17; _tf = 0.010; _sf = 0.10; _L_fl = 19.0
_nf     = int(round(_ch_w / _sf))   # fins per channel

def finned_floor(q_b_W, T_in_C=40.0, T_plate_max_C=90.0, A_cool=None):
    """Find min-fan-power air velocity for finned floor carrying flux q_b_W [W/m2].
    Returns dict or None if T_plate > T_plate_max_C at all velocities tried.
    Key outputs: V [m/s], heff [W/m2/K], Vd [m3/s], T_pl_out [C], dp [Pa], P_fan [W].
    """
    if A_cool is None: A_cool = A_foot
    best = None
    for V in np.arange(1.0, 15.1, 0.1):
        w   = _sf - _tf; Ach = w * _H_ch; P_wet = 2*_Hf + 2*w; Dh = 4*Ach / P_wet
        Re  = V * Dh / _nu_a
        h   = 0.023 * Re**0.8 * _Pr_a**0.4 * _ka / Dh
        m_f = np.sqrt(2*h / (_kst * _tf))
        eta = np.tanh(m_f * _Hf) / (m_f * _Hf)
        heff = h * (_ch_w - _nf*_tf + _nf*2*_Hf*eta) / _rail_p
        nch  = (A_cool / _L_fl) / _rail_p
        Vd   = V * nch * (_ch_w - _nf*_tf) * _H_ch; mdot = _rho_a * Vd
        q_air = q_b_W * 0.95
        dT_air = q_air * A_cool / (mdot * _cp_a)
        T_out  = T_in_C + dT_air
        T_pl_in  = T_in_C  + q_air / heff
        T_pl_out = T_out + q_air / heff
        f_f  = (0.79 * np.log(Re) - 1.64)**(-2)
        dp   = (f_f * _L_fl / Dh + 2.0) * 0.5 * _rho_a * V**2 + 80   # +80 Pa fittings
        P_fan = dp * Vd / 0.60
        if T_pl_out <= T_plate_max_C and (best is None or P_fan < best['P_fan']):
            best = dict(V=V, Re=Re, h=h, eta=eta, heff=heff, Vd=Vd, mdot=mdot,
                        dT_air=dT_air, T_out=T_out, T_pl_in=T_pl_in,
                        T_pl_out=T_pl_out, dp=dp, P_fan=P_fan)
    return best

print(f"Finned floor: {_nf} fins/ch @ {_sf*1000:.0f} mm  "
      f"fin {_tf*1000:.0f}x{_Hf*1000:.0f} mm  "
      f"ch {_ch_w*1000:.0f}x{_H_ch*1000:.0f} mm  "
      f"rail {_rail_w*1000:.0f}@{_rail_p*1000:.0f} mm  L={_L_fl:.0f} m")
print(f"  A_foot = {A_foot:.1f} m2   T_in = {T_amb_C:.0f} C   plate limit 90/93 C")
print()
print(f"  {'q_b':>5}  {'V':>5}  {'Re':>6}  {'h':>5}  {'eta':>5}  {'heff':>5}  "
      f"{'Vd':>5}  {'dTair':>6}  {'Tpl_in':>7}  {'Tpl_out':>8}  {'dP':>5}  {'Fan':>6}")

_best_q = _best_all['q_b'] if '_best_all' in dir() and _best_all else 1000
for _qb in sorted(set([_best_q, 400, 700, 1000, 1500, 2000])):
    rf = finned_floor(_qb, T_in_C=T_amb_C, T_plate_max_C=90.0)
    if rf is None:
        print(f"  {_qb:5.0f} W/m2  -> infeasible (plate > 90 C at all velocities)")
    else:
        print(f"  {_qb:5.0f}  {rf['V']:5.1f}  {rf['Re']:6.0f}  {rf['h']:5.0f}  "
              f"{rf['eta']:5.2f}  {rf['heff']:5.0f}  {rf['Vd']:5.1f}  "
              f"{rf['dT_air']:6.1f}K  {rf['T_pl_in']:7.1f}C  {rf['T_pl_out']:8.1f}C  "
              f"{rf['dp']:5.0f}Pa  {rf['P_fan']/1e3:5.1f}kW")

# Structural bearing check (from floor.py weight table)
_W = dict(graphite=m_graphite/1e3,
          thermax_bottom=A_foot * 2.0 * TH_RHO / 1e3,   # 2 m bottom est.
          cols_posts=150, shell_roof=450,
          floor_plate=A_foot * 0.020 * 7850 / 1e3,
          fins=104, deck_drives=600)
_Wt = sum(_W.values())
print(f"\nBearing loads: total ~{_Wt:.0f} t  ->  "
      f"avg {_Wt*9.81/A_foot:.0f} kPa on floor  |  "
      f"rail bearing ~{_Wt*9.81/A_foot/(_rail_w/_rail_p):.0f} kPa")
